# Using crocoddyl to solve optimal control problems

Before using this notebook, you will have to install [crocoddyl](https://github.com/loco-3d/crocoddyl). You can do this either by calling `conda install crocoddyl` **from the command line** inside your conda environment (do **not** install it inside this notebook), or by recreating your conda environment using the `environment.yml` file in the course github repository — this file now contains the dependency on crocoddyl.

## Set up the notebook

Do imports.

In [4]:
# Rigid body dynamics (pinocchio)
import pinocchio as pin

# Collision checking
import coal

# Optimal control
import crocoddyl

# Visualization (meshcat)
from pinocchio.visualize import MeshcatVisualizer
import meshcat_shapes

# Robot models (robot_descriptions)
from robot_descriptions.loaders.pinocchio import load_robot_description

# Math
import numpy as np

# Timing
import time

# Plots
import matplotlib.pyplot as plt

# Suppress the display of very small numbers
np.set_printoptions(suppress=True)

## Load a robot

Load a robot model.

In [5]:
robot = load_robot_description(
    'iiwa14_description',           # name of robot model
    root_joint=None,                # fixed base
)

Enable checking for self-collision.

In [6]:
# Add all pairs
robot.collision_model.addAllCollisionPairs()

# List useless collision pairs
useless_pairs = [0, 7, 13, 18, 22, 25, 26, 27]

# Remove useless collision pairs
for i in sorted(useless_pairs, reverse=True):
    robot.collision_model.removeCollisionPair(robot.collision_model.collisionPairs[i])

Define function to convert each primitive geometry to a convex mesh by sampling its surface.

In [7]:
def primitive_to_convex(shape, n=32):
    """
    Convert a coal primitive shape to a ConvexBase by sampling
    its surface, building a BVHModel from the points, and computing
    the convex hull.
    """
    if isinstance(shape, coal.Box):
        # half-extents
        x, y, z = shape.halfSide
        verts = np.array([
            [ x,  y,  z], [ x,  y, -z], [ x, -y,  z], [ x, -y, -z],
            [-x,  y,  z], [-x,  y, -z], [-x, -y,  z], [-x, -y, -z],
        ])

    elif isinstance(shape, coal.Cylinder):
        r = shape.radius
        h = shape.halfLength
        theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
        circle = np.column_stack([r * np.cos(theta), r * np.sin(theta)])
        top = np.column_stack([circle, np.full(n, h)])
        bot = np.column_stack([circle, np.full(n, -h)])
        verts = np.vstack([top, bot])

    elif isinstance(shape, coal.Cone):
        r = shape.radius
        h = shape.halfLength
        theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
        base = np.column_stack([r * np.cos(theta), r * np.sin(theta),
                                np.full(n, -h)])
        apex = np.array([[0.0, 0.0, h]])
        verts = np.vstack([base, apex])

    elif isinstance(shape, coal.Sphere):
        verts = _sphere_points(shape.radius, n)

    elif isinstance(shape, coal.Ellipsoid):
        rx, ry, rz = shape.radii
        unit = _sphere_points(1.0, n)
        verts = unit * np.array([rx, ry, rz])

    elif isinstance(shape, coal.Capsule):
        r = shape.radius
        h = shape.halfLength
        # hemisphere + cylinder ring
        unit = _sphere_points(1.0, n)
        top_hemi = unit[unit[:, 2] >= 0] * r + [0, 0, h]
        bot_hemi = unit[unit[:, 2] <= 0] * r + [0, 0, -h]
        theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
        ring = np.column_stack([r * np.cos(theta), r * np.sin(theta),
                                np.zeros(n)])
        verts = np.vstack([top_hemi, bot_hemi,
                           ring + [0, 0, h], ring + [0, 0, -h]])

    else:
        raise TypeError(f"Unsupported shape type: {type(shape)}")

    return _points_to_convex(verts)


def _sphere_points(radius, n):
    """Fibonacci sphere sampling."""
    indices = np.arange(n * n // 2)  # enough points
    # Use a simpler grid approach
    u = np.linspace(0, 2 * np.pi, n, endpoint=False)
    v = np.linspace(0, np.pi, n // 2 + 1)
    theta, phi = np.meshgrid(u, v)
    theta, phi = theta.ravel(), phi.ravel()
    x = radius * np.sin(phi) * np.cos(theta)
    y = radius * np.sin(phi) * np.sin(theta)
    z = radius * np.cos(phi)
    return np.column_stack([x, y, z])


def _points_to_convex(points):
    """Build a BVHModel from a point cloud and compute its convex hull."""
    bvh = coal.BVHModelOBBRSS()
    bvh.beginModel(0, len(points))
    bvh.addVertices(points)
    bvh.endModel()
    bvh.buildConvexHull(True, "Qt")
    return bvh.convex

Apply function to convert all collision geometries to convex meshes.

In [8]:
# Choose sampling density
n = 32

# Apply function
for geom_obj in robot.collision_model.geometryObjects:
    geom = geom_obj.geometry
    if isinstance(geom, coal.ConvexBase):
        print(f'No need to convert "{geom_obj.name}" (already a convex mesh)')
        continue
    elif isinstance(geom, coal.BVHModelBase):
        print(f'Convert "{geom_obj.name}" (buildConvexHull)')
        geom.buildConvexHull(True, "Qt")
        geom_obj.geometry = geom.convex
    elif isinstance(geom, coal.ShapeBase):
        print(f'Convert "{geom_obj.name}" (primitive_to_convex)')
        geom_obj.geometry = primitive_to_convex(geom, n=n)
    else:
        print(f'Not sure what to do with "{geom_obj.name}"')

Convert "iiwa_link_0_0" (primitive_to_convex)
Convert "iiwa_link_1_0" (primitive_to_convex)
Convert "iiwa_link_2_0" (primitive_to_convex)
Convert "iiwa_link_3_0" (primitive_to_convex)
Convert "iiwa_link_4_0" (primitive_to_convex)
Convert "iiwa_link_5_0" (primitive_to_convex)
Convert "iiwa_link_6_0" (buildConvexHull)
Convert "iiwa_link_7_0" (buildConvexHull)


Make all collision objects semi-transparent.

In [9]:
for go in robot.collision_model.geometryObjects:
    go.meshColor[-1] = 0.3

for go in robot.visual_model.geometryObjects:
    go.meshColor[-1] = 0.7

Update robot data.

In [10]:
robot.data = robot.model.createData()
robot.collision_data = robot.collision_model.createData()
robot.visual_data = robot.visual_model.createData()

Display robot (and obstacles) in browser.

In [11]:
# Create a visualizer
vis = MeshcatVisualizer(robot.model, robot.collision_model, robot.visual_model)
robot.setVisualizer(vis, init=False)
vis.initViewer(open=True)
vis.loadViewerModel()

# Pause to allow meshcat initialization to finish
time.sleep(0.1)

# Choose what to display
vis.displayFrames(False)
vis.displayVisuals(True)
vis.displayCollisions(True)

# Add our own frames to the visualizer because the default frames are hard to see
frames_to_show = [
    'iiwa_link_0',
    'iiwa_link_1',
    'iiwa_link_2',
    'iiwa_link_3',
    'iiwa_link_4',
    'iiwa_link_5',
    'iiwa_link_6',
    'iiwa_link_7',
    'iiwa_link_ee_kuka',
]
for frame in frames_to_show:
    meshcat_shapes.frame(
        vis.viewer['frames/' + frame],
        opacity=0.5,
        axis_length=0.1,
        axis_thickness=0.0025,
        origin_radius=0.005,
    )

# Pause to allow meshcat initialization to finish
time.sleep(0.1)

# Change the camera view
vis.setCameraPosition(np.array([0., 2., 0.75]))
vis.setCameraTarget(np.array([0., 0., 0.75]))

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7005/static/


/usr/bin/xdg-open: 882: x-www-browser: not found
/usr/bin/xdg-open: 882: firefox: not found
/usr/bin/xdg-open: 882: iceweasel: not found
/usr/bin/xdg-open: 882: seamonkey: not found
/usr/bin/xdg-open: 882: mozilla: not found
/usr/bin/xdg-open: 882: epiphany: not found
/usr/bin/xdg-open: 882: konqueror: not found


Put the robot at its "neutral" configuration.

In [12]:
# Get and show the neutral configuration (most likely all zeros)
q = pin.neutral(robot.model)
print(f'{q = }')

# Do forward kinematics
# - Compute the placement of all joint frames (modifies robot.data but not robot.model or q)
pin.forwardKinematics(robot.model, robot.data, q)
# - Compute the placement of all link frames (modifies robot.data but not robot.model)
pin.updateFramePlacements(robot.model, robot.data)
# - Compute the placement of all collision geometry objects
pin.updateGeometryPlacements(
    robot.model,
    robot.data,
    robot.collision_model,
    robot.collision_data,
)

# Show the configuration in the visualizer
vis.display(q)
for frame in frames_to_show:
    frame_id = robot.model.getFrameId(frame)
    vis.viewer['frames/' + frame].set_transform(robot.data.oMf[frame_id].homogeneous)

q = array([0., 0., 0., 0., 0., 0., 0.])


/usr/bin/xdg-open: 882: chromium: not found


## Define and solve a single optimal control problem

Choose and display a desired pose.

In [13]:
# Choose a desired pose
target_position = np.array([0.4, 0., 0.8])
target_orientation = np.eye(3)
target_pose = pin.SE3(target_orientation, target_position)

# Show desired pose in the browser window
meshcat_shapes.frame(vis.viewer['frames/' + 'desired_pose'], opacity=1.0, axis_length=0.2)
vis.viewer['frames/' + 'desired_pose'].set_transform(target_pose.homogeneous)

Check that all limits are finite.

In [14]:
with np.printoptions(precision=2):
    print('ALL LIMITS MUST BE FINITE')
    print(f' lowerPositionLimit: {robot.model.lowerPositionLimit}')
    print(f' upperPositionLimit: {robot.model.upperPositionLimit}')
    print(f'      velocityLimit: {robot.model.velocityLimit}')
    print(f'        effortLimit: {robot.model.effortLimit}')

ALL LIMITS MUST BE FINITE
 lowerPositionLimit: [-2.97 -2.09 -2.97 -2.09 -2.97 -2.09 -3.05]
 upperPositionLimit: [2.97 2.09 2.97 2.09 2.97 2.09 3.05]
      velocityLimit: [1.48 1.48 1.75 1.31 2.27 2.36 2.36]
        effortLimit: [320. 320. 176. 176. 110.  40.  40.]


/usr/bin/xdg-open: 882: chromium-browser: not found
/usr/bin/xdg-open: 882: google-chrome: not found
/usr/bin/xdg-open: 882: www-browser: not found
/usr/bin/xdg-open: 882: links2: not found
/usr/bin/xdg-open: 882: elinks: not found
/usr/bin/xdg-open: 882: links: not found
/usr/bin/xdg-open: 882: lynx: not found
/usr/bin/xdg-open: 882: w3m: not found
xdg-open: no method available for opening 'http://127.0.0.1:7005/static/'


Define an optimal control problem and create a solver for this problem.

In [15]:
# Parameters
nv = robot.model.nv
nu = nv
dt = 1e-2
T = 100

# State and actuation models
state = crocoddyl.StateMultibody(robot.model)
actuation = crocoddyl.ActuationModelFull(state)

# Reference for state regularization
q0 = pin.neutral(robot.model)
x0 = np.concatenate([q0, pin.utils.zero(nv)])

# Running costs that are the same for all time steps
# - Input regularization
uRegRes = crocoddyl.ResidualModelJointEffort(state, actuation, nu)
uRegCost = crocoddyl.CostModelResidual(state, uRegRes)
# - State regularization
xRegAct = crocoddyl.ActivationModelWeightedQuad(
    np.concatenate((1e-1 * np.ones(nv), 1e0 * np.ones(nv)))
)
xRegRes = crocoddyl.ResidualModelState(state, x0, nu)
xRegCost = crocoddyl.CostModelResidual(state, xRegAct, xRegRes)

# Running models
runningModels = []
target_id = state.pinocchio.getFrameId('iiwa_link_ee_kuka')
for i in range(T):
    # Running cost at time step i
    trackRes_i = crocoddyl.ResidualModelFramePlacement(
        state,
        target_id,
        pin.SE3.Identity(), # <-- placeholder
        nu,
    )
    trackCost_i = crocoddyl.CostModelResidual(state, trackRes_i)
    
    # Total running cost at time step i (sum of individual costs)
    costModel_i = crocoddyl.CostModelSum(state)
    costModel_i.addCost('track', trackCost_i, 1e1)
    costModel_i.addCost('xReg', xRegCost, 1e-3)
    costModel_i.addCost('uReg', uRegCost, 1e-6)

    # Dynamic model at time step i (no external contact constraints)
    dam_i = crocoddyl.DifferentialActionModelFreeFwdDynamics(
        state, actuation, costModel_i,
    )

    # Running model at time step i (use Euler integration)
    runningModel_i = crocoddyl.IntegratedActionModelEuler(dam_i, dt)
    runningModels.append(runningModel_i)

# Terminal cost
# - State regularization
xRegAct_T = crocoddyl.ActivationModelWeightedQuad(
    np.concatenate((1e-1 * np.ones(nv), 1e0 * np.ones(nv)))
)
xRegRes_T = crocoddyl.ResidualModelState(state, x0, nu)
xRegCost_T = crocoddyl.CostModelResidual(state, xRegAct_T, xRegRes_T)
# - Tracking
trackRes_T = crocoddyl.ResidualModelFramePlacement(
    state,
    target_id,
    pin.SE3.Identity(), # <-- placeholder
    nu,
)
trackCost_T = crocoddyl.CostModelResidual(state, trackRes_T)

# Terminal model
costModel_T = crocoddyl.CostModelSum(state)
costModel_T.addCost('track', trackCost_T, 1e1)
costModel_T.addCost('xReg', xRegCost_T, 1e-3)
terminalModel = crocoddyl.IntegratedActionModelEuler(
    crocoddyl.DifferentialActionModelFreeFwdDynamics(
        state, actuation, costModel_T,
    ),
    0.,
)

# Problem and solver
problem = crocoddyl.ShootingProblem(x0, runningModels, terminalModel)
solver = crocoddyl.SolverFDDP(problem)

# Make the solver verbose
solver.setCallbacks([crocoddyl.CallbackVerbose()])

Modify the optimal control problem to set the target pose.

In [ ]:
# Function that returns the target pose at (continuous) time t
def compute_target(t):
    # FIXME
    return target_pose

# Set target pose for each runningModel and for the terminalModel
t_current = 0.
for i in range(T):
    t = t_current + i * dt
    runningModels[i].differential.costs.costs['track'].cost.residual.reference = compute_target(t)
terminalModel.differential.costs.costs['track'].cost.residual.reference = compute_target(t_current + T * dt)

Solve optimal control problem.

In [ ]:
# Solve problem
success = solver.solve(
    [],  # <-- guess at state trajectory
    [],  # <-- guess at input trajetory
    200, # <-- maximum number of iterations
)

# Check for success
assert(success)

# Put results in a log
log = {
    't': np.array([t_current + i * dt for i in range(T + 1)]),
    'x': np.array(solver.xs),
    'u': np.array(solver.us),
}

Look at the size of `t`, `x`, `u` in log (notice that `u` has length one less than the other two).

In [ ]:
print(f'len(t) = {len(log['t'])}')
print(f'len(x) = {len(log['x'])}')
print(f'len(u) = {len(log['u'])}')

Define functions to show and plot results.

In [ ]:
def show_results(log):
    qs = log['x'][:, :nv]
    t_start = time.time()
    for i, q in enumerate(qs):
        t_target = t_start + i * dt
        vis.display(q)
        t_now = time.time()
        if t_now < t_target:
            time.sleep(t_target - t_now)

def plot_results(log):
    fig = plt.figure(figsize=(18, 6))
    for i in range(nv):
        ax = fig.add_subplot(3, nv, i + 1)
        ax.plot(log['t'], log['x'][:, i], label=rf'$q_{i}$')
        ax.legend()
        ax = fig.add_subplot(3, nv, nv + (i + 1))
        ax.plot(log['t'], log['x'][:, nv + i], label=rf'$v_{i}$')
        ax.legend()
        ax = fig.add_subplot(3, nv, (2 * nv) + (i + 1))
        ax.plot(log['t'][:-1], log['u'][:, i], label=rf'$u_{i}$')
        ax.legend()
    fig.tight_layout()
    plt.show()

Show and plot results.

In [ ]:
show_results(log)
plot_results(log)

## Wrap the solver in an MPC loop

Remove solver callbacks.

In [ ]:
solver.setCallbacks([])

Implement MPC loop.

In [ ]:
def compute_target(t):
    # FIXME
    return target_pose

x_current = x0.copy()
xs_warm = []
us_warm = []

# Log
log = {
    't': [],
    'x': [x_current.copy()],
    'u': [],
}

K = 500 # <-- 5 seconds at dt = 1e-2
for k in range(K):
    # Get current time
    t_current = k * dt
    log['t'].append(t_current)

    # Update problem
    problem.x0 = x_current
    for i in range(T):
        t = t_current + i * dt
        runningModels[i].differential.costs.costs['track'].cost.residual.reference = compute_target(t)
    terminalModel.differential.costs.costs['track'].cost.residual.reference = compute_target(t_current + T * dt)

    # Solve problem
    solver.solve(xs_warm, us_warm, 200)
    converged = solver.isFeasible and solver.stop < solver.th_stop
    assert(converged)

    # Get control input
    u_current = solver.us[0].copy()
    log['u'].append(u_current)

    # Apply control input to simulate one time step
    # FIXME
    x_current = solver.xs[1].copy()
    log['x'].append(x_current)

    # Copy and shift the solution for next warm-start
    xs_warm = [solver.xs[i].copy() for i in range(1, T + 1)] + [solver.xs[T].copy()]
    us_warm = [solver.us[i].copy() for i in range(1, T)]     + [solver.us[T - 1].copy()]

# Append final time to log
log['t'].append(K * dt)

# Clean up log
for k in log.keys():
    log[k] = np.array(log[k])

Show and plot results.

In [ ]:
show_results(log)
plot_results(log)

## How to add other constraints

The code cells shown below are not meant to be evaluated. They provide snippets of code that you could incorporate into the example OCP/MPC shown above. The code cells below are also not meant to be exhaustive — you may need or want to add other constraints or other terms to the cost in your own implementation.

### Joint angle and joint velocity limits (soft constraints)

Here is how to create a cost model that penalizes violating these limits with a quadratic barrier function. To use it, you would need to add this cost to each running model and to the terminal model.

In [ ]:
# Activation bounds are the limits beyond which a quadratic penalty will be applied
xLim = crocoddyl.ActivationBounds(
    np.concatenate([robot.model.lowerPositionLimit, -robot.model.velocityLimit]), # <-- lower bounds
    np.concatenate([robot.model.upperPositionLimit,  robot.model.velocityLimit]), # <-- upper bounds
    beta=1., # <-- if you make this smaller, then the penalty starts even before you exceed the limits
)

# The activation model is a quadratic barrier function
xLimAct = crocoddyl.ActivationModelQuadraticBarrier(xLim)

# Residual and cost follow the same pattern as before (with the reference being all zeros)
xLimRes = crocoddyl.ResidualModelState(state, np.zeros(state.nx), nu)
xLimCost = crocoddyl.CostModelResidual(state, xLimAct, xLimRes)

To see what a quadratic barrier function is, do this and read the paragraph that starts "The activation is zero when..."

In [ ]:
help(crocoddyl.ActivationModelQuadraticBarrier)

### Joint torque limits (hard constraints)

Currently, joint torque limits are the only hard constraints that can be handled by crocoddyl (other than the equality constraints that define the dynamics). It is easy to add them — simply replace the two lines that define the problem and solver as follows.

First, add attributes to each running model that define upper and lower bounds on joint torque (this is done in the loop the creates all of the running models).

In [ ]:
runningModel_i.u_lb = -robot.model.effortLimit
runningModel_i.u_ub =  robot.model.effortLimit

Second, add the same attributes to the terminal model (this is done just before creating the problem and the solver).

In [ ]:
terminalModel.u_lb = -robot.model.effortLimit
terminalModel.u_ub =  robot.model.effortLimit

Finally, replace the line of code that creates the solver with the following line of code. The "BoxFDDP" solver is what can handle this particular hard constraint.

In [ ]:
solver = crocoddyl.SolverBoxFDDP(problem)

### Collision avoidance (soft constraints)

Here is how to create a cost model that penalizes collision avoidance with a quadratic barrier function.

In [ ]:
# List to hold the cost models
collisionCosts = []

# Append a cost model to the list for each collision pair
for i, pair in enumerate(robot.collision_model.collisionPairs):
    # Activation
    collisionAct = crocoddyl.ActivationModel2NormBarrier(
        3,      # <-- residual has three components (distance to collision in 3D)
        0.02,   # <-- quadratic barrier begins when distance is less than this threshold (meters)
    )

    # Residual
    collisionRes = crocoddyl.ResidualModelPairCollision(
        state,
        nu,
        robot.collision_model,
        i,          # index of the collision pair
        pair.first, # index of first geometry object (determines which frame is used)
    )

    # Cost
    collisionCost = crocoddyl.CostModelResidual(state, collisionAct, collisionRes)

    # Append to list
    collisionCosts.append()

Here is how you might add these costs to each running model. (You would need to add these same costs to the terminal model.)

In [ ]:
for k in range(len(robot.collision_model.collisionPairs)):
    costModel_i.addCost(f'selfCollision_{k}', collisionCosts[k], 1e1)

Here is the activation model (i.e., the barrier function) we used. It is more appropriate for collision avoidance than the activation model we suggested to enforce joint angle and velocity limits.

In [ ]:
help(crocoddyl.ActivationModel2NormBarrier)